# ChiralFold Quick Demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/ChiralFold_Quick_Demo.ipynb)

**ChiralFold** — a chirality-preserving peptide structure toolkit that guarantees 0% D-peptide chirality violations, compared to AlphaFold 3's 51% violation rate.

This notebook demonstrates the three core workflows:
1. **PDB audit** — chirality, Ramachandran, planarity, clashes
2. **D-peptide conformer generation** — 0% chirality violation by construction
3. **Mirror-image L↔D PDB transformation** — exact, RMSD = 0.0 Å

**Reference:** Childs CM, Zhou P, Donald BR (2025). *Has AlphaFold 3 Solved the Protein Folding Problem for D-Peptides?* bioRxiv 2025.03.14.643307. [doi:10.1101/2025.03.14.643307](https://doi.org/10.1101/2025.03.14.643307)

In [ ]:
# Cell 1 — Install ChiralFold
!pip install git+https://github.com/Tommaso-R-Marena/ChiralFold.git

In [ ]:
# Cell 2 — Verify installation and import the public API
import chiralfold
from chiralfold import (
    audit_pdb, format_report,            # PDB auditing
    ChiralFold,                          # D-peptide prediction
    mirror_pdb,                          # L↔D transformation
    detect_chirality_violations,         # AF3 correction
    correct_chirality,
)
print('ChiralFold version:', chiralfold.__version__)

In [ ]:
# Cell 3 — Demo 1: Audit a PDB structure
import urllib.request
urllib.request.urlretrieve('https://files.rcsb.org/download/1LDF.pdb', '1LDF.pdb')

report = audit_pdb('1LDF.pdb')
print('=== DEMO 1: PDB AUDIT ===')
print(format_report(report))

# Demo 2: Generate D-peptide conformers (16/16 converged in production)
model = ChiralFold()
pred = model.predict('AFWKELDR')
n_converged = sum(1 for c in pred.get('conformers', []) if c['converged'])
print('\n=== DEMO 2: D-PEPTIDE PREDICTION ===')
print(f"Sequence    : {pred['sequence']}")
print(f"Chirality   : {pred['chirality_pattern']}")
print(f"Violations  : {pred['chirality_violations']} / {pred['n_chiral_residues']} ({pred['violation_rate']*100:.1f}%)")
print(f"Conformers  : {len(pred.get('conformers', []))} generated, {n_converged} converged")

# Demo 3: Mirror-image PDB transformation (L → D)
mirror_result = mirror_pdb('1LDF.pdb', '1LDF_D_mirror.pdb')
print('\n=== DEMO 3: MIRROR-IMAGE TRANSFORMATION ===')
print(f"Mirrored {mirror_result['n_atoms']} atoms across {mirror_result['n_residues']} residues")
print(f"Output : {mirror_result['output_path']}")

## Expected Output

```
=== AUDIT ===
════════════════════════════════════════════════════════════
ChiralFold PDB Auditor — Quality Report
════════════════════════════════════════════════════════════
  Residues : 260    Atoms : 1948    Score : 81.4 / 100
── Cα Chirality ──  Correct : 222  Wrong : 0  (100.0%)
── Ramachandran ──  Favored : 95.6%  Outlier : 0.0%
── Planarity ─────  Within 6°: 100.0%  Outliers : 0

=== PREDICTION ===
Sequence   : AFWKELDR
Chirality  : DDDDDDDD
Violations : 0 / 8 (0.0%)
Conformers : 16 generated, 16 converged
Best energy: -118.3440 kcal/mol (conf_id=5)

=== VIOLATIONS DETECTED ===
Checked    : 209 residues
Violations : 0 (0.0%)
```

## Citation

If you use ChiralFold in your work, please cite:

```bibtex
@software{chiralfold2026,
  title   = {ChiralFold: General-Purpose Protein Stereochemistry Toolkit},
  author  = {Tommaso R. Marena},
  year    = {2026},
  url     = {https://github.com/Tommaso-R-Marena/ChiralFold},
  version = {3.2.1}
}
```
